# Testscript for Master Thesis 

## TOC 

- [Setup](#setup)
- [Load and Sample Tweets for Labeling](#load_tweets)
- [Define the different prompts](#define_prompts)
- [Run the Labeling Tasks (Single)](#run_labeling_single)
- [Run the Labeling Tasks (Single) + Confidence](#run_labeling_single_conf)
- [Run the Labeling Tasks (Batch)](#run_labeling_batch)
- [Run the Labeling Tasks (Batch) + Confidence](#run_labeling_batch_confidence)

#### Open ToDo's
-Clean tweets from nbsps etc.  
-rewrite data frame creation and data collection into separate functions  
-maybe do the batch partition and order setup at the beginning of the script   
    --> then we can use the exactly same df for all upcoming steps 

In [423]:
# General Setup and Git
# cd to path of current script
import os
import sys
os.chdir(sys.path[0])

! git pull

Already up to date.


In [422]:
# Git codes
## see git status
! git status # see git status

# Ask the user for a commit message
commit_message = input("Enter commit message: ")

## Git add the current stuff to commit and set commit message
! git add .
! git commit -m "$commit_message"

## Push commits to remote
! git push

On branch master
Your branch is up to date with 'origin/master'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .ipynb_checkpoints/4a_GPT_Labeling-checkpoint.ipynb
	modified:   4a_GPT_Labeling.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


Enter commit message:  updates


[master 8bb147a] updates
 2 files changed, 86 insertions(+), 34 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 8 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 931 bytes | 931.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To github.com:T-Reiter/master-thesis.git
   04ea459..8bb147a  master -> master


<a id="setup"></a>
### SANDBOX & SETUP

In [2]:
# Load API key from gitignored api_key module 
from api_key import API_KEY as API_KEY

In [11]:
# base Request
# test capabilties
from openai import OpenAI

# Setup open AI client using api key
client = OpenAI(
  api_key=API_KEY
)

# Test prompt
return_number = "Return me a number between 1 and 100. Only return the number"

# Test completion
completion = client.chat.completions.create(
  model="gpt-4o-mini",
  store=True,
  messages=[
    {"role": "user", 
     "content": return_number
    }
  ],
  temperature = 1 # default temp = 1  
)

print("Response: ", completion.choices[0].message.content);

Response:  73


<a id="load_tweets"></a>
### Load and Sample Tweets for Labeling

#### Load the Tweets from the Kern et al. (2023) paper

In [118]:
# load the full tweets as used by CK
import pandas as pd
# pd.set_option('max_colwidth', 1000)
pd.set_option('display.max_colwidth', 0)

# Load the tweet files
folder_path = 'Tweets_CK/'
train_file = 'full_train_s.csv'
test_file = 'full_test_s.csv'

train_df = pd.read_csv(os.path.join(folder_path, train_file))
test_df = pd.read_csv(os.path.join(folder_path, test_file))

# Reorder test_df columns to match train_df
test_df = test_df[train_df.columns]

# Add the 'original_split' column
train_df['original_split'] = 'train'
test_df['original_split'] = 'test'

# Combine them
tweets_full = pd.concat([train_df, test_df], ignore_index=True)

# Rename columns for a consistent style 
tweets_full = (tweets_full.
               rename(columns={'tweet.id': 'tweet_id', 
                               'batch.tweet': 'batch_tweet',
                               'hate.speech': 'hate_speech', 
                               'offensive.language': 'offensive_language'}))

In [13]:
# Get Basic Info 
num_rows = tweets_full.shape[0]
num_cols = tweets_full.shape[1]
num_unique_tweets = tweets_full['tweet_id'].nunique()
num_batches = tweets_full['batch_tweet'].nunique()
tweet_counts = (tweets_full[['tweet_hashed','batch_tweet']]
                .drop_duplicates().groupby('batch_tweet').size())
average_tweets_per_batch = tweet_counts.mean()

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Number of unique tweet IDs: {num_unique_tweets}")
print(f"Number of Batches: {num_batches}")
print(f'Average Tweets per Batch: {average_tweets_per_batch}')
tweets_full.sort_values('tweet_id').head()

Number of rows: 44900
Number of columns: 8
Number of unique tweet IDs: 3000
Number of Batches: 50
Average Tweets per Batch: 60.0


,tweet_id,id,version,batch_tweet,hate_speech,offensive_language,tweet_hashed,original_split
2927,1,7,B,R1,0.0,1.0,@###### bro that hoe live,train
2930,1,13,A,R1,0.0,1.0,@###### bro that hoe live,train
2932,1,155,A,R1,0.0,1.0,@###### bro that hoe live,train
2918,1,125,E,R1,1.0,0.0,@###### bro that hoe live,train
2919,1,173,E,R1,0.0,1.0,@###### bro that hoe live,train


#### Clean tweets from escape characters etc.

In [ ]:
# TO DO?

#### Randomly Sample N tweets and create the result df out of them

In [125]:
# randomly sample N unique tweets with their original tweet id
N_SAMPLES = 3

sampled_tweets = (
    tweets_full
    .drop_duplicates(subset='tweet_hashed')
    .sample(n=N_SAMPLES
            # ,random_state=42 # no seed to get different cases during dev
           )
    [['tweet_id', 'tweet_hashed']]
)

sampled_tweets.head()

,tweet_id,tweet_hashed
22484,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people
19552,598,people be shooting their guns around here late as fuck got me thinking they tryna shoot my black ass 4 jammin jiggaboo music
21647,543,@###### and who said I didn't like myself where the fuck u got shit from u made that up stfu and stop assuming and bitch am not ugly.


<a id="define_prompts"></a>
### Define the different Prompts

In [376]:
# Define the prompts

PROMPT_DICT = {
    # SINGLE TWEET PROMPTING
    ### A: Both OL and HS at the same time, OL first
    "A":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.OL: Only OL
    "C.OL":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.HS: Only HS
    "C.HS":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""",
    
    
    # SINGLE TWEET PROMPTING + CONFIDENCE SCORE
    ### A: Both OL and HS at the same time, OL first
    "A_conf":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B_conf":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""", 
    ### C.OL: Only OL
    "C.OL_conf":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""", 
    ### C.HS: Only HS
    "C.HS_conf":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""",
    
    
    # SINGLE TWEET PROMPTING WITH EXPLANATION
    "A_ex":"""You will see a series of tweets. For each tweet, we want you to code if it contains hate speech or contains offensive language.
    We define hate speech as:  Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate speech. 
    We define offensive language as:  Language that is highly offensive to certain individuals or groups but does not meet the requirements of hate speech.
    Some tweets contains both hate speech AND offensive language.
    
    For each tweet, we want you to code whether it contains Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B_ex":"""You will see a series of tweets. For each tweet, we want you to code if it contains hate speech or contains offensive language.
    We define hate speech as:  Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate speech. 
    We define offensive language as:  Language that is highly offensive to certain individuals or groups but does not meet the requirements of hate speech.
    Some tweets contains both hate speech AND offensive language.
    
    For each tweet, we want you to code whether it contains Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.OL: Only OL
    "C_ex.OL":"""You will see a series of tweets. For each tweet, we want you to code if it contains contains offensive language.
    We define offensive language as:  Language that is highly offensive to certain individuals or groups.
    
    For each tweet, we want you to code whether it contains Offensive Language (OL).
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.HS: Only HS
    "C_ex.HS":"""You will see a series of tweets. For each tweet, we want you to code if it contains contains hate speech.
    We define hate speech as:  Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.
    
    For each tweet, we want you to code whether it contains Hate Speech (HS).
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""",
    

    # BATCH PROMPTING (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    For each tweet return only one label.
    Separate the the chosen labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### C.OL: Only OL
    "C.OL_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### C.HS: Only HS
    "C.HS_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}"""    
}

<a id="run_labeling_single"></a>
### Run the Labeling Tasks (Single Tweet Prompting)

#### Create Result DF

In [131]:
# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of the ones in the prompt dictionary)

CONDITIONS = ['A', 'B']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"response_{i}"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"confidence_{i}"] = ""    
    
results.head()

,tweet_id,tweet,condition,response_1,response_2,response_3,confidence_1,confidence_2,confidence_3
0,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,A,,,,,,
1,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,B,,,,,,
2,598,people be shooting their guns around here late as fuck got me thinking they tryna shoot my black ass 4 jammin jiggaboo music,A,,,,,,
3,598,people be shooting their guns around here late as fuck got me thinking they tryna shoot my black ass 4 jammin jiggaboo music,B,,,,,,
4,543,@###### and who said I didn't like myself where the fuck u got shit from u made that up stfu and stop assuming and bitch am not ugly.,A,,,,,,


In [132]:
%%time 

# Run the actual Loop for API Calls 

for row in range(0,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt = PROMPT_DICT[condition].format(tweet=tweet_to_rate)
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=1,
            store=True,
        )
        label = resp.choices[0].message.content.strip()
        # print(prompt)
        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"response_{response}"] = label

n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results

Number of API Calls: 18
CPU times: user 189 ms, sys: 14.6 ms, total: 204 ms
Wall time: 11.1 s


,tweet_id,tweet,condition,response_1,response_2,response_3,confidence_1,confidence_2,confidence_3
0,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,A,OLHS,OLHS,OLHS,,,
1,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,B,HS,HS,HS,,,
2,598,people be shooting their guns around here late as fuck got me thinking they tryna shoot my black ass 4 jammin jiggaboo music,A,OLHS,OLHS,OLHS,,,
3,598,people be shooting their guns around here late as fuck got me thinking they tryna shoot my black ass 4 jammin jiggaboo music,B,HSOL,HSOL,HSOL,,,
4,543,@###### and who said I didn't like myself where the fuck u got shit from u made that up stfu and stop assuming and bitch am not ugly.,A,OL,OL,OL,,,
5,543,@###### and who said I didn't like myself where the fuck u got shit from u made that up stfu and stop assuming and bitch am not ugly.,B,OL,OL,OL,,,


In [163]:
# save result file 
import os 
from datetime import date

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = "100t_4c_3r"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_single_conf"></a>
### Run the Labeling Task (with confidence scores)

#### Create Result Df

In [134]:
# ISSUE: Loop is executed sequential s.t. we cannot use the acctual query-per-sec quote of the API (3r/s)
# SOL: --> rewrite code s.t. it's parallelized

# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_conf', 'B_conf', 'C.OL_conf', 'C.HS_conf']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_Token1"] = ""   
    results[f"R{i}_Token2"] = ""  
    results[f"R{i}_Token3"] = ""      

    
results.head()

,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score,R1_Token1,R1_Token2,R1_Token3,R2_Token1,R2_Token2,R2_Token3,R3_Token1,R3_Token2,R3_Token3
0,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,A_conf,,,,,,,,,,,,,,,
1,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,B_conf,,,,,,,,,,,,,,,
2,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,C.OL_conf,,,,,,,,,,,,,,,
3,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,C.HS_conf,,,,,,,,,,,,,,,
4,598,people be shooting their guns around here late as fuck got me thinking they tryna shoot my black ass 4 jammin jiggaboo music,A_conf,,,,,,,,,,,,,,,


#### Run the Loop and score confidence scores

In [136]:
tweet_id_to_rate = results.iloc[1]['tweet_id']
tweet_to_rate = results.iloc[1]['tweet']
condition = results.iloc[0]['condition']
prompt = PROMPT_DICT[condition].format(tweet=tweet_to_rate)
print(tweet_id)
print(tweet_to_rate)
print(condition)

543
@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people
A_conf


In [154]:
%%time 

# Run the actual Loop for API Calls 
N_TOP_LOGPROBS = 5

for row in range(0,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt = PROMPT_DICT[condition].format(tweet=tweet_to_rate)
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp_full = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=1,
            logprobs=True,
            top_logprobs= N_TOP_LOGPROBS,
            store=True,
        )
        # Extract the Labels & Confidence Score
        resp = resp_full.choices[0].message.content.strip()
        resp_split = resp.split(',')
        label = resp_split[0]
        score = resp_split[1].strip()
        
        # Extract the Token Logprobs for tokens 1 to 3
        # exract logprobs for first token
        top_k_logprobs_first = resp_full.choices[0].logprobs.content[0].top_logprobs
        top_n_tokens_first = [obj.token for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        top_n_logprobs_first = [obj.logprob for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        # token_value_dict = dict(zip(top_n_tokens_first, top_n_logprobs_first))
        token_value_series_first = pd.Series(top_n_logprobs_first, index=top_n_tokens_first)

        # exract logprobs for second token
        top_k_logprobs_second = resp_full.choices[0].logprobs.content[1].top_logprobs
        top_n_tokens_second = [obj.token for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        top_n_logprobs_second = [obj.logprob for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        token_value_series_second = pd.Series(top_n_logprobs_second, index=top_n_tokens_second)

        # exract logprobs for third token
        top_k_logprobs_third = resp_full.choices[0].logprobs.content[2].top_logprobs
        top_n_tokens_third = [obj.token for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
        top_n_logprobs_third = [obj.logprob for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        token_value_series_third = pd.Series(top_n_logprobs_third, index=top_n_tokens_third)

        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"R{response}_label"] = label
        results.at[row, f"R{response}_score"] = score
        results.at[row, f"R{response}_Token1"] = token_value_series_first
        results.at[row, f"R{response}_Token2"] = token_value_series_second
        results.at[row, f"R{response}_Token3"] = token_value_series_third
        
    # Print Progress
    print(f"""Finished Row {row+1} of {results.shape[0]} --> {round((row+1)/results.shape[0]*100,1)} %""", 
          end='\r')


n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(1)

Number of API Calls: 36-> 100.0 %
CPU times: user 423 ms, sys: 24.8 ms, total: 447 ms
Wall time: 17.4 s


,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score,R1_Token1,R1_Token2,R1_Token3,R2_Token1,R2_Token2,R2_Token3,R3_Token1,R3_Token2,R3_Token3
0,160,@###### @###### @###### they are wrong you are a nigger always causung mischief among ur own people,A_conf,OLHS,OLHS,OLHS,4,4,4,"OL -0.000912 * -7.000912 OL -15.250912 AL -17.375912 *, -17.750912 dtype: float64","HS 0.000 , -17.250 H -18.125 HS -19.625 Hs -20.875 dtype: float64",", 0.000 , -17.250 ', -18.125 ， -19.625 -20.875 dtype: float64","OL -0.002476 * -6.002476 OL -15.127476 ** -16.877476 *, -17.127476 dtype: float64","HS 0.000 , -17.500 H -17.875 HS -20.250 Hs -20.750 dtype: float64",", 0.000 , -17.500 ', -17.875 ， -20.250 ,, -20.750 dtype: float64","OL -0.001929 * -6.251929 OL -15.376929 *, -16.501928 ** -17.376928 dtype: float64","HS 0.000 , -17.125 H -19.750 hs -21.625 HS -21.750 dtype: float64",", 0.000 , -17.125 ', -19.750 ， -21.625 -21.750 dtype: float64"


<a id="run_labeling_batch"></a>
### Run the Labeling Tasks (Batch Prompting)

#### Create Random Batches
In the original papers, the batch creation process does not seem to be included in the code.  
However, the tweets were randomly grouped into batches of 50.  
It is unclear whether this grouping was done based on the original tweet id gathered from elsewhere or whether the IDs were enumerated afterwards.  
In the end, we have have 50 tweets per batch (60 batches accordingly) that always appeared in the same order per batch.  
This randomized batch allocation process is replicated in the following.

In [377]:
# Randomly partition the df into batches reorder the tweets within batches 
BATCH_SIZE = 10

seed = 13
tweets_full_shuffled = (tweets_full[['tweet_id','tweet_hashed']]
                        .drop_duplicates()
                        .sample(frac=1, random_state=seed)
                        .reset_index(drop=True))
# create batch IDs
tweets_full_shuffled['batch_id'] = (tweets_full_shuffled.index // BATCH_SIZE) + 1
# create position IDs for each tweet within its batch
tweets_full_shuffled['tweet_in_batch'] = (tweets_full_shuffled.index % BATCH_SIZE) + 1

tweets_full_shuffled.head(5)

,tweet_id,tweet_hashed,batch_id,tweet_in_batch
0,811,Fat fucking funky nasty ass hoes,1,1
1,742,Up early then a bitch driving to denton omg can I move already,1,2
2,2098,@###### @###### there isn't a green one either. There's red and yellow in that pic...,1,3
3,2349,The only thing about niccas is they be followers smh .. Be ya self everyone else is taken!!,1,4
4,1371,@###### I'm the type to put that bitch in a river so you can gtf out my locations lol,1,5


#### Sample Random Batches

In [341]:
# sample radnom batches
N_BATCH_SAMPLES = 2
sampled_batches = (pd.Series(tweets_full_shuffled['batch_id']
                   .unique())
                   .sample(n=N_BATCH_SAMPLES
                           , random_state=13
                          ))

# get the tweets in these batches 
batch_sampled_tweets = (tweets_full_shuffled[tweets_full_shuffled['batch_id']
                                    .isin(sampled_batches)]
                       [['batch_id','tweet_in_batch', 'tweet_id','tweet_hashed']]
                        .drop_duplicates())

print(f'Number of Batches Sampled: {N_BATCH_SAMPLES}')
print(f"Number of Unique Tweets Sampled: {len(batch_sampled_tweets['tweet_hashed'])}")
print(f'Sampled Batch IDs: {sampled_batches.to_list()}')
batch_sampled_tweets.iloc[0:3]

Number of Batches Sampled: 2
Number of Unique Tweets Sampled: 20
Sampled Batch IDs: [115, 100]


,batch_id,tweet_in_batch,tweet_id,tweet_hashed
990,100,1,1657,@###### Roberts is a fuckin fag&#128514; smack the shit outta him
991,100,2,137,"@###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot."
992,100,3,2587,@###### Waco is ghetto hahaha. Wait til we go pick up nat in Austin tomorrow


#### Create Results DF

In [342]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch', 'B_batch']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in batch_sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"response_{i}"] = ""

results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,response_1,response_2,response_3
0,100,1,1657,@###### Roberts is a fuckin fag&#128514; smack the shit outta him,A_batch,,,
1,100,1,1657,@###### Roberts is a fuckin fag&#128514; smack the shit outta him,B_batch,,,
2,100,2,137,"@###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot.",A_batch,,,
3,100,2,137,"@###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot.",B_batch,,,
4,100,3,2587,@###### Waco is ghetto hahaha. Wait til we go pick up nat in Austin tomorrow,A_batch,,,


#### Run the Loop

In [374]:
%%time 

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 1

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt = PROMPT_DICT[condition].format(tweet=tweets_in_batch_str)

        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 1):
            # get response via API Call
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=1,
                store=True,
            )
            labels = resp.choices[0].message.content.strip()
            labels_list = labels.split(';')
            labels_list

        
            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"response_{response}"] = labels_list[i]
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(5)

Number of API Calls: 40
CPU times: user 109 ms, sys: 9.73 ms, total: 118 ms
Wall time: 9.72 s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,response_1,response_2,response_3
0,100,1,1657,@###### Roberts is a fuckin fag&#128514; smack the shit outta him,A_batch,OLHS,OLHS,OLHS
1,100,1,1657,@###### Roberts is a fuckin fag&#128514; smack the shit outta him,B_batch,HSOL,HSOL,HSOL
2,100,2,137,"@###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot.",A_batch,OLHS,OLHS,OLHS
3,100,2,137,"@###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot.",B_batch,HSOL,HSOL,HSOL
4,100,3,2587,@###### Waco is ghetto hahaha. Wait til we go pick up nat in Austin tomorrow,A_batch,none,none,none


<a id="run_labeling_batch_confidence"></a>
### Run the Labeling Tasks (Batch Prompting) + Self-Report Confidence Scores

In [395]:
# Define the prompts

PROMPT_DICT = {
    # SINGLE TWEET PROMPTING
    ### A: Both OL and HS at the same time, OL first
    "A":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.OL: Only OL
    "C.OL":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    ### C.HS: Only HS
    "C.HS":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""",
    
    
    # SINGLE TWEET PROMPTING + CONFIDENCE SCORE
    ### A: Both OL and HS at the same time, OL first
    "A_conf":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B_conf":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""", 
    ### C.OL: Only OL
    "C.OL_conf":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""", 
    ### C.HS: Only HS
    "C.HS_conf":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    Please return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Seperate them by ','. Do not add any further explanations.
    Here is the tweet: {tweet}""",
    

    # BATCH PROMPTING (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    For each tweet return only one label.
    Separate the the chosen labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### C.OL: Only OL
    "C.OL_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### C.HS: Only HS
    "C.HS_batch":"""You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}""",
    
    
    # batch prompting with confidence scores
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch_conf":"""You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    For each tweet,only return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Separate label and confidence score by ','.
    Separate the responses for the different tweets by ';'
    Do not add any further explanations.
    Here are the tweets: {tweet}""", 
    ### B: Both OL and HS at the same time, HS first
    "B_batch_conf":"""You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### C.OL: Only OL
    "C.OL_batch_conf":"""You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}""", 
    ### C.HS: Only HS
    "C.HS_batch_conf":"""You are given multiple tweets and for each of them your task is to tell whether it contains Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Separate the labels for the different tweets with ';'
    Here are the tweets: {tweet}"""    
}

#### Create Result Df (use the sample sampled batches as before)

In [403]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch_conf']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in batch_sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    

results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
0,100,1,1657,@###### Roberts is a fuckin fag&#128514; smack the shit outta him,A_batch_conf,,,,,,
1,100,2,137,"@###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot.",A_batch_conf,,,,,,
2,100,3,2587,@###### Waco is ghetto hahaha. Wait til we go pick up nat in Austin tomorrow,A_batch_conf,,,,,,
3,100,4,1191,@###### you styl gotta get dem tweets off my nig,A_batch_conf,,,,,,
4,100,5,2338,Gay ass nigga &#128514;&#128514; bitch you thought you thought,A_batch_conf,,,,,,


#### Run the Loop

In [404]:
batchID = 100
condition = 'A_batch_conf'
tweets_in_batch = (results[results['batch_id'] == batchID]
                   ['tweet']
                   .unique())
tweets_in_batch_str = "\n" + "\n".join(
    [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
)
prompt = (PROMPT_DICT[condition]
          .format(tweet=tweets_in_batch_str))

resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=1,
            store=True,
        )

response = resp.choices[0].message.content.strip()
responses = response.split(';')
responses

print(response)
# labels = resp.choices[0].message.content.strip()
# labels_list = labels.split(';')
# labels_list

OL, 4; OLHS, 4; none, 3; HS, 4; HS, 4; none, 2; none, 2; HS, 3; none, 2; HS, 4


In [410]:
response = resp.choices[0].message.content.strip()
responses = response.split(';')
responses_per_tweet = [response.split(',') for response in responses]
responses_per_tweet
labels = [tweet_response[0].strip() 
          for tweet_response in responses_per_tweet]
scores = [tweet_response[1].strip() for tweet_response in responses_per_tweet]
print(prompt)
print(labels)
print(scores)
labels[1]

You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    In addition to the tweet label also provide a confidence score between 1 and 4 for your labeling.
    Here, 1 reresents lowest confidence and 4 represents highest confidence. 
    For each tweet,only return only the parts in between the *asterisks* (without asterisks) and the confidence score.
    Separate label and confidence score by ','.
    Separate the responses for the different tweets by ';'
    Do not add any further explanations.
    Here are the tweets: 
1) @###### Roberts is a fuckin fag&#128514; smack the shit outta him
2) @###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot.
3) @###### Waco is ghetto hahaha. Wait til w

'OLHS'

In [411]:
%%time 

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 1

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt = PROMPT_DICT[condition].format(tweet=tweets_in_batch_str)

        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 3):
            # get response via API Call
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=1,
                store=True,
            )
            api_response_full = resp.choices[0].message.content.strip()
            api_responses = api_response_full.split(';')
            api_responses_per_tweet = [api_response.split(',') for api_response in api_responses]
            # Extract the Labels & Confidence Scores per tweet
            labels_list = [tweet_response[0].strip() 
                           for tweet_response in api_responses_per_tweet]
            scores_list = [tweet_response[1].strip() 
                           for tweet_response in api_responses_per_tweet]

        
            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"R{response}_label"] = labels_list[i]
                results.at[idx, f"R{response}_score"] = scores_list[i]
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(5)

Number of API Calls: 20
CPU times: user 79.3 ms, sys: 22.3 ms, total: 102 ms
Wall time: 7.39 s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
0,100,1,1657,@###### Roberts is a fuckin fag&#128514; smack the shit outta him,A_batch_conf,OL,OL,OL,4,3,4
1,100,2,137,"@###### Aww y u so mad tho, a successful man LMFAO, hut hugging as faggot.",A_batch_conf,OL,OL,OLHS,4,3,3
2,100,3,2587,@###### Waco is ghetto hahaha. Wait til we go pick up nat in Austin tomorrow,A_batch_conf,none,none,none,3,4,4
3,100,4,1191,@###### you styl gotta get dem tweets off my nig,A_batch_conf,HS,HS,HS,4,4,4
4,100,5,2338,Gay ass nigga &#128514;&#128514; bitch you thought you thought,A_batch_conf,HS,HS,HS,4,4,4


# Notes 
1) The tweets can be processed in parts to either (e.g., the first 2k for all conditions)  
A) be able to split the task into free sub parts or   
B) to be able to better estimate the total costs  

2) Probably the Batch Calls have to be done in a separate code snippet

3) Costs: 
4o-mini : 
Input:          15ct  /1M Tokens
Cached Input:   7.5ct /1M Tokens (might be our case)
Output:         60ct  /1M Tokens


In [136]:
# Get Token Counts 
from utils import count_tokens
import statistics
# count tokens for gpt-3.5-turbo model (default in tiktoken package 
print(count_tokens(PROMPT_DICT['A'])) # 111 TOKENS for standard condition prompt 
print(statistics.fmean([count_tokens(x) for x in sampled_tweets])) # 15 Average Tweet Token length
# response has max 2 tokens

111
15.8
